# TA-005D — GAN Augmentation para Clases Minoritarias

**Objetivo:** Entrenar una DCGAN por cada clase minoritaria del dataset VetXRay y generar imágenes sintéticas para balancear el conjunto de entrenamiento.

**Clases a aumentar:**
- `bronchial_pattern`: 454 reales → objetivo 1000 (+546 sintéticas)
- `alveolar_pattern`: 671 reales → objetivo 1000 (+329 sintéticas)
- `pleural_effusion`: 283 reales → objetivo 700 (+417 sintéticas)

**Output:** `train_augmented.csv` con rutas a imágenes reales + sintéticas

**Monitoreo:** W&B proyecto `vetxray-cnn`, run `TA-005D-GAN-Augmentation`

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import pydicom
import wandb
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms
from pathlib import Path
from tqdm.notebook import tqdm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

BASE          = Path(r'E:\Taller Integrador I\ModelosIA\Radiografias')
CHECKPOINTS   = Path(r'E:\Taller Integrador I\ModelosIA\modelos\checkpoints')
NOTEBOOKS_DIR = Path(r'E:\Taller Integrador I\ModelosIA\modelos\Notebooks')
SYNTHETIC_DIR = BASE / 'dataset_split' / 'synthetic'
TRAIN_CSV     = BASE / 'dataset_split' / 'manifests' / 'train.csv'
TRAIN_DIR     = BASE / 'dataset_split' / 'train'

SYNTHETIC_DIR.mkdir(parents=True, exist_ok=True)

CLASSES = ['alveolar_pattern', 'bronchial_pattern', 'pleural_effusion', 'cardiomegaly', 'no_finding']

NOISE_DIM    = 100
GAN_IMG_SIZE = 128
GAN_EPOCHS   = 300
GAN_BATCH    = 16
LR_G         = 2e-4
LR_D         = 2e-4
SEED         = 42

AUG_TARGETS = {
    'bronchial_pattern': 546,
    'alveolar_pattern':  329,
    'pleural_effusion':  417,
}

torch.manual_seed(SEED)
np.random.seed(SEED)

df_train = pd.read_csv(TRAIN_CSV)
print(f'\nTrain set: {len(df_train)} imagenes')
for c in CLASSES:
    n = df_train['TAG'].str.contains(c, na=False).sum()
    print(f'  {c}: {n}')

In [ ]:
wandb.init(
    project='vetxray-cnn',
    entity='dbaylont1-antenor-orrego-private-university',
    name='TA-005D-GAN-Augmentation',
    config={
        'task': 'TA-005D',
        'architecture': 'DCGAN',
        'noise_dim': NOISE_DIM,
        'img_size': GAN_IMG_SIZE,
        'epochs': GAN_EPOCHS,
        'batch_size': GAN_BATCH,
        'lr_G': LR_G,
        'lr_D': LR_D,
        'aug_targets': AUG_TARGETS,
        'seed': SEED
    },
    tags=['GAN', 'augmentation', 'TA-005D', 'VetXRay']
)
print('W&B run iniciado:', wandb.run.url)

In [ ]:
def load_dicom_for_gan(path, img_size=GAN_IMG_SIZE):
    dcm = pydicom.dcmread(str(path))
    img = dcm.pixel_array.astype(np.float32)
    if hasattr(dcm, 'PhotometricInterpretation') and dcm.PhotometricInterpretation == 'MONOCHROME1':
        img = img.max() - img
    p2, p98 = np.percentile(img, 2), np.percentile(img, 98)
    img = np.clip(img, p2, p98)
    img = (img - p2) / (p98 - p2 + 1e-8)
    img_pil = Image.fromarray((img * 255).astype(np.uint8))
    img_pil = img_pil.resize((img_size, img_size), Image.LANCZOS)
    tensor = transforms.ToTensor()(img_pil)
    tensor = (tensor - 0.5) / 0.5
    return tensor


class GANDataset(Dataset):
    def __init__(self, df, class_name, train_dir, img_size=GAN_IMG_SIZE):
        mask = df['TAG'].str.contains(class_name, na=False)
        self.files = [train_dir / row['FileName'] for _, row in df[mask].iterrows()]
        self.img_size = img_size
        print(f'  {class_name}: {len(self.files)} imagenes reales para GAN')

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        return load_dicom_for_gan(self.files[idx], self.img_size)

In [ ]:
class Generator(nn.Module):
    def __init__(self, noise_dim=NOISE_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(noise_dim, 256 * 8 * 8),
            nn.Unflatten(1, (256, 8, 8)),
            nn.BatchNorm2d(256),
            nn.ReLU(True),
            nn.ConvTranspose2d(256, 128, 4, 2, 1),
            nn.BatchNorm2d(128),
            nn.ReLU(True),
            nn.ConvTranspose2d(128, 64, 4, 2, 1),
            nn.BatchNorm2d(64),
            nn.ReLU(True),
            nn.ConvTranspose2d(64, 32, 4, 2, 1),
            nn.BatchNorm2d(32),
            nn.ReLU(True),
            nn.ConvTranspose2d(32, 1, 4, 2, 1),
            nn.Tanh()
        )

    def forward(self, z):
        return self.net(z)


class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 32, 4, 2, 1),
            nn.LeakyReLU(0.2, True),
            nn.Conv2d(32, 64, 4, 2, 1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2, True),
            nn.Conv2d(64, 128, 4, 2, 1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, True),
            nn.Conv2d(128, 256, 4, 2, 1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, True),
            nn.Flatten(),
            nn.Linear(256 * 8 * 8, 1)
        )

    def forward(self, x):
        return self.net(x)


def init_weights(m):
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d, nn.Linear)):
        nn.init.normal_(m.weight, 0.0, 0.02)
    if isinstance(m, nn.BatchNorm2d):
        nn.init.normal_(m.weight, 1.0, 0.02)
        nn.init.constant_(m.bias, 0)


g_test = Generator()
d_test = Discriminator()
z_test = torch.randn(1, NOISE_DIM)
out_g  = g_test(z_test)
out_d  = d_test(out_g)
print(f'Generator output shape:     {out_g.shape}')
print(f'Discriminator output shape: {out_d.shape}')
del g_test, d_test, z_test, out_g, out_d

In [ ]:
def train_gan(class_name, df, epochs=GAN_EPOCHS):
    print(f'\n{"="*55}')
    print(f'Entrenando GAN: {class_name}')

    dataset = GANDataset(df, class_name, TRAIN_DIR)
    loader  = DataLoader(dataset, batch_size=GAN_BATCH, shuffle=True,
                         num_workers=0, pin_memory=False, drop_last=True)

    G = Generator().to(DEVICE)
    D = Discriminator().to(DEVICE)
    G.apply(init_weights)
    D.apply(init_weights)

    opt_G     = torch.optim.Adam(G.parameters(), lr=LR_G, betas=(0.5, 0.999))
    opt_D     = torch.optim.Adam(D.parameters(), lr=LR_D, betas=(0.5, 0.999))
    criterion = nn.BCEWithLogitsLoss()

    fixed_noise = torch.randn(16, NOISE_DIM, device=DEVICE)

    for epoch in range(epochs):
        d_losses, g_losses = [], []

        for real_imgs in loader:
            real_imgs   = real_imgs.to(DEVICE)
            bs          = real_imgs.size(0)
            real_labels = torch.ones(bs, 1, device=DEVICE)
            fake_labels = torch.zeros(bs, 1, device=DEVICE)

            opt_D.zero_grad()
            d_real = criterion(D(real_imgs), real_labels)
            z      = torch.randn(bs, NOISE_DIM, device=DEVICE)
            d_fake = criterion(D(G(z).detach()), fake_labels)
            d_loss = (d_real + d_fake) / 2
            d_loss.backward()
            opt_D.step()

            opt_G.zero_grad()
            z      = torch.randn(bs, NOISE_DIM, device=DEVICE)
            g_loss = criterion(D(G(z)), real_labels)
            g_loss.backward()
            opt_G.step()

            d_losses.append(d_loss.item())
            g_losses.append(g_loss.item())

        avg_d = np.mean(d_losses)
        avg_g = np.mean(g_losses)

        log_dict = {
            f'{class_name}/d_loss': avg_d,
            f'{class_name}/g_loss': avg_g,
            f'{class_name}/epoch':  epoch + 1
        }

        if (epoch + 1) % 25 == 0:
            G.eval()
            with torch.no_grad():
                samples = (G(fixed_noise) * 0.5 + 0.5).clamp(0, 1)
            log_dict[f'{class_name}/samples'] = [
                wandb.Image(samples[i].cpu(), caption=f'ep{epoch+1}') for i in range(8)
            ]
            G.train()
            print(f'  Epoch {epoch+1:3d}/{epochs} | D: {avg_d:.4f} | G: {avg_g:.4f}')

        wandb.log(log_dict)

    ckpt_path = CHECKPOINTS / f'gan_generator_{class_name}.pth'
    torch.save(G.state_dict(), ckpt_path)
    print(f'  Guardado: {ckpt_path.name}')
    return G

In [ ]:
def generate_synthetic(G, class_name, n_images):
    G.eval()
    out_dir = SYNTHETIC_DIR / class_name
    out_dir.mkdir(parents=True, exist_ok=True)

    saved    = []
    batch_sz = 32
    generated = 0

    with torch.no_grad():
        while generated < n_images:
            n    = min(batch_sz, n_images - generated)
            z    = torch.randn(n, NOISE_DIM, device=DEVICE)
            imgs = (G(z) * 0.5 + 0.5).clamp(0, 1)

            for i, img in enumerate(imgs):
                fname = f'synthetic_{class_name}_{generated + i:04d}.png'
                fpath = out_dir / fname
                img_np = (img.squeeze().cpu().numpy() * 255).astype(np.uint8)
                Image.fromarray(img_np).save(fpath)
                saved.append(str(fpath))

            generated += n

    print(f'  Generadas {n_images} imagenes → {out_dir}')
    return saved

In [ ]:
synthetic_records = []

for class_name, n_gen in AUG_TARGETS.items():
    G     = train_gan(class_name, df_train)
    paths = generate_synthetic(G, class_name, n_gen)

    for p in paths:
        synthetic_records.append({
            'FileName':       Path(p).name,
            'PatientName':    'synthetic',
            'breed':          'unknown',
            'specie':         'unknown',
            'Projection':     'unknown',
            'Quality':        'correct',
            'TAG':            class_name,
            'NOTE':           'GAN_synthetic',
            'split':          'train',
            'is_synthetic':   True,
            'synthetic_path': p
        })

    wandb.log({f'{class_name}/total_synthetic': n_gen})
    del G
    torch.cuda.empty_cache()

print('\nEntrenamiento GAN completado para todas las clases.')

In [ ]:
df_train['is_synthetic']   = False
df_train['synthetic_path'] = None

df_synthetic = pd.DataFrame(synthetic_records)
df_augmented = pd.concat([df_train, df_synthetic], ignore_index=True)

aug_csv = BASE / 'dataset_split' / 'manifests' / 'train_augmented.csv'
df_augmented.to_csv(aug_csv, index=False)

print(f'Guardado: {aug_csv}')
print(f'Total train augmented: {len(df_augmented)}')
print(f'  Reales:     {len(df_train)}')
print(f'  Sinteticas: {len(df_synthetic)}')
print()

counts_after = {}
for c in CLASSES:
    n = df_augmented['TAG'].str.contains(c, na=False).sum()
    counts_after[c] = n
    print(f'  {c}: {n}')

wandb.log({'augmented_class_counts': counts_after})

table = wandb.Table(columns=['clase', 'reales', 'sinteticas', 'total'])
counts_real = {}
for c in CLASSES:
    counts_real[c] = df_train['TAG'].str.contains(c, na=False).sum()
for c in CLASSES:
    n_syn = df_synthetic['TAG'].str.contains(c, na=False).sum() if not df_synthetic.empty else 0
    table.add_data(c, counts_real[c], n_syn, counts_after[c])
wandb.log({'resumen_augmentation': table})

In [ ]:
wandb.finish()
print('W&B run cerrado.')
print(f'\nTA-005D completado.')
print(f'Imagenes sinteticas: {SYNTHETIC_DIR}')
print(f'Manifest augmented:  {aug_csv}')
print('\nProximo paso: TA-003B / TA-004B / TA-005E — reentrenar los 3 modelos con train_augmented.csv')